# Mounter

> Mounts directories as static files for serving through the web server.

In [ ]:
#| default_exp serving.mounter

In [ ]:
#| export
import hashlib
from pathlib import Path
from typing import Dict, List, Optional
from fastcore.basics import patch

from starlette.staticfiles import StaticFiles
from starlette.routing import Mount

## DirectoryMounter

Mounts directories for static file serving with instance-level state. Uses Starlette's `StaticFiles` middleware for efficient serving with proper MIME types, caching headers, and range request support.

In [ ]:
#| export
class DirectoryMounter:
    """Mounts directories for static file serving with instance-level state."""

    # Prefix pattern for this mounter's routes
    ROUTE_PREFIX_PATTERN = "mg_static_"

    def __init__(self):
        """Initialize the mounter with empty state."""
        self._mounted: Dict[str, str] = {}  # directory -> route_prefix
        self._app = None

In [ ]:
#| export
@patch
def mount(
    self: DirectoryMounter,
    app,  # FastHTML/Starlette application instance
    directories: List[str]  # List of directory paths to mount
) -> None:
    """Mount directories to app for static file serving."""
    self._app = app
    self._mounted.clear()

    # Remove any existing mounts from this instance
    self._remove_existing_mounts(app)

    for directory in directories:
        self._mount_directory(app, directory)

In [ ]:
#| export
@patch
def get_url(
    self: DirectoryMounter,
    file_path: str  # Full path to the file
) -> Optional[str]:  # URL to access the file, or None if not in a mounted directory
    """Get URL for a file based on mounted directories."""
    file_path_resolved = Path(file_path).resolve()

    for directory, prefix in self._mounted.items():
        dir_path = Path(directory).resolve()
        try:
            relative_path = file_path_resolved.relative_to(dir_path)
            return f"/{prefix}/{relative_path.as_posix()}"
        except ValueError:
            # File is not in this directory
            continue

    return None

In [ ]:
#| export
@patch
def is_mounted(
    self: DirectoryMounter,
    directory: str  # Directory path to check
) -> bool:  # True if the directory is mounted
    """Check if a directory is currently mounted."""
    return directory in self._mounted

In [ ]:
#| export
@patch
def get_mounted_directories(
    self: DirectoryMounter
) -> List[str]:  # List of mounted directory paths
    """Get list of currently mounted directories."""
    return list(self._mounted.keys())

In [ ]:
#| export
@patch
def unmount_all(
    self: DirectoryMounter
) -> None:
    """Remove all mounts from this instance."""
    if self._app:
        self._remove_existing_mounts(self._app)
    self._mounted.clear()

In [ ]:
#| export
@patch
def _mount_directory(
    self: DirectoryMounter, 
    app,  # FastHTML/Starlette application instance
    directory: str  # Directory path to mount
) -> None:
    """Mount a single directory."""
    dir_path = Path(directory)

    if not dir_path.exists():
        print(f"[DirectoryMounter] Warning: Directory does not exist: {directory}")
        return

    if not dir_path.is_dir():
        print(f"[DirectoryMounter] Warning: Path is not a directory: {directory}")
        return

    prefix = self._generate_prefix(directory)

    try:
        mount = Mount(
            f"/{prefix}",
            app=StaticFiles(directory=str(dir_path)),
            name=prefix
        )

        # Insert at the beginning of routes (before other routes)
        app.routes.insert(0, mount)

        # Store the mapping
        self._mounted[directory] = prefix

    except Exception as e:
        print(f"[DirectoryMounter] Error mounting {directory}: {e}")

In [ ]:
#| export
@patch
def _generate_prefix(
    self: DirectoryMounter, 
    directory: str  # Directory path
) -> str:  # Route prefix string (e.g., "mg_static_abc12345")
    """Generate a unique route prefix for a directory using MD5 hash."""
    dir_hash = hashlib.md5(directory.encode()).hexdigest()[:8]
    return f"{self.ROUTE_PREFIX_PATTERN}{dir_hash}"

In [ ]:
#| export
@patch
def _remove_existing_mounts(
    self: DirectoryMounter, 
    app  # FastHTML/Starlette application instance
) -> None:
    """Remove existing mounts matching this mounter's prefix pattern."""
    # Filter out mounts with our prefix pattern
    app.routes[:] = [
        route for route in app.routes
        if not (
            isinstance(route, Mount) and
            route.path.startswith(f"/{self.ROUTE_PREFIX_PATTERN}")
        )
    ]

In [ ]:
#| export
@patch
def create_url_getter(
    self: DirectoryMounter
) -> callable:  # Function that converts path to URL
    """Create a URL getter function for use with gallery components."""
    def get_url(path: str) -> Optional[str]:
        return self.get_url(path)
    return get_url

## Tests

In [ ]:
import tempfile

# Simple mock app for testing
class MockApp:
    def __init__(self):
        self.routes = []

# Test DirectoryMounter initialization
mounter = DirectoryMounter()
assert mounter._mounted == {}
assert mounter._app is None
assert mounter.ROUTE_PREFIX_PATTERN == "mg_static_"

print("DirectoryMounter initialization tests passed!")

DirectoryMounter initialization tests passed!


In [ ]:
# Test prefix generation
mounter = DirectoryMounter()
prefix = mounter._generate_prefix("/some/path")
assert prefix.startswith("mg_static_")
assert len(prefix) == len("mg_static_") + 8  # 8 char hash

# Same path should give same prefix
prefix2 = mounter._generate_prefix("/some/path")
assert prefix == prefix2

# Different path should give different prefix
prefix3 = mounter._generate_prefix("/other/path")
assert prefix3 != prefix

print("Prefix generation tests passed!")

Prefix generation tests passed!


In [ ]:
# Test mounting directories
with tempfile.TemporaryDirectory() as tmpdir:
    # Create test structure
    subdir = Path(tmpdir) / "media"
    subdir.mkdir()
    test_file = subdir / "video.mp4"
    test_file.write_text("test content")
    
    app = MockApp()
    mounter = DirectoryMounter()
    
    # Mount directory
    mounter.mount(app, [str(subdir)])
    
    # Verify mount was added to app routes
    assert len(app.routes) == 1
    assert isinstance(app.routes[0], Mount)
    assert app.routes[0].path.startswith("/mg_static_")
    
    # Verify internal state
    assert mounter.is_mounted(str(subdir))
    assert str(subdir) in mounter.get_mounted_directories()
    
    # Test get_url
    url = mounter.get_url(str(test_file))
    assert url is not None
    assert url.startswith("/mg_static_")
    assert "video.mp4" in url
    
    # File not in mount should return None
    other_file = Path(tmpdir) / "other.txt"
    other_file.write_text("other")
    assert mounter.get_url(str(other_file)) is None

print("Mount and URL tests passed!")

Mount and URL tests passed!


In [ ]:
# Test multiple mounts and unmount_all
with tempfile.TemporaryDirectory() as tmpdir:
    dir1 = Path(tmpdir) / "dir1"
    dir2 = Path(tmpdir) / "dir2"
    dir1.mkdir()
    dir2.mkdir()
    
    app = MockApp()
    mounter = DirectoryMounter()
    
    # Mount multiple directories
    mounter.mount(app, [str(dir1), str(dir2)])
    assert len(app.routes) == 2
    assert len(mounter.get_mounted_directories()) == 2
    
    # Unmount all
    mounter.unmount_all()
    assert len(app.routes) == 0
    assert len(mounter.get_mounted_directories()) == 0

print("Multiple mounts and unmount tests passed!")

Multiple mounts and unmount tests passed!


In [ ]:
# Test remounting clears old mounts
with tempfile.TemporaryDirectory() as tmpdir:
    dir1 = Path(tmpdir) / "dir1"
    dir2 = Path(tmpdir) / "dir2"
    dir1.mkdir()
    dir2.mkdir()
    
    app = MockApp()
    mounter = DirectoryMounter()
    
    # First mount
    mounter.mount(app, [str(dir1)])
    assert len(app.routes) == 1
    assert mounter.is_mounted(str(dir1))
    
    # Remount with different directory - should clear old mount
    mounter.mount(app, [str(dir2)])
    assert len(app.routes) == 1  # Old mount removed, new one added
    assert not mounter.is_mounted(str(dir1))
    assert mounter.is_mounted(str(dir2))

print("Remount cleanup tests passed!")

Remount cleanup tests passed!


In [ ]:
# Test URL generation for nested files
with tempfile.TemporaryDirectory() as tmpdir:
    # Create nested structure
    nested = Path(tmpdir) / "media" / "videos" / "2024"
    nested.mkdir(parents=True)
    test_file = nested / "clip.mp4"
    test_file.write_text("test")
    
    app = MockApp()
    mounter = DirectoryMounter()
    mounter.mount(app, [str(Path(tmpdir) / "media")])
    
    # URL should include full relative path
    url = mounter.get_url(str(test_file))
    assert url is not None
    assert "videos/2024/clip.mp4" in url

print("Nested file URL tests passed!")

Nested file URL tests passed!


In [ ]:
# Test create_url_getter
with tempfile.TemporaryDirectory() as tmpdir:
    subdir = Path(tmpdir) / "media"
    subdir.mkdir()
    (subdir / "video.mp4").write_text("test")
    
    app = MockApp()
    mounter = DirectoryMounter()
    mounter.mount(app, [str(subdir)])
    
    # Create getter function
    get_url = mounter.create_url_getter()
    
    # Test getter
    url = get_url(str(subdir / "video.mp4"))
    assert url is not None
    assert "video.mp4" in url
    
    # Test file not in mount
    assert get_url("/some/other/path.txt") is None

print("create_url_getter tests passed!")

create_url_getter tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()